In [1]:
# === repo-root bootstrap (added during the experiments/ reorganization) ===
# Locate the repository root (the directory containing `core/`), switch the
# working directory there, and put it on sys.path. This lets the notebook
# import `core`/`execution`/`analysis` and resolve every `shared_data/...`
# path regardless of which directory the notebook was launched from.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "core")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
assert os.path.isdir(os.path.join(_root, "core")), "repo root (with core/) not found"
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /Users/henryzou/Documents/GitHub/OQPA


# QPA Fidelity Decay -- DD Method Comparison -- Unrolled (T=1) -- IBM Pittsburgh

Spin-off of [`end_to_end_unrolled_pittsburgh_t1.ipynb`](end_to_end_unrolled_pittsburgh_t1.ipynb), focused on **one question: which dynamical decoupling (DD) sequence best closes the gap to theory?**

The parent notebook ran a 2-point DD sweep -- DD off vs. DD on with the `XpXm` sequence. Here that axis is widened into a **method sweep**:

| Method | Sequence | Where it runs |
|--------|----------|---------------|
| `off`  | -- (no DD) | -- |
| `XpXm` | `tau/2 - (+X) - tau - (-X) - tau/2` | runtime (server-side) -- **the parent notebook's default** |
| `XX`   | `tau/2 - (+X) - tau - (+X) - tau/2` | runtime (server-side) |
| `XY4`  | `tau/2 - X - tau - Y - tau - X - tau - Y - tau/2` | runtime (server-side) |
| `UDD`  | Uhrig DD -- n X-pulses at non-uniform times | **manual** (client-side transpiler pass) |

`XX`, `XpXm`, `XY4` are the three sequences the IBM runtime exposes via `sampler.options.dynamical_decoupling.sequence_type`. **UDD is not a runtime built-in**, so it is padded into the circuits ourselves with qiskit's `PadDynamicalDecoupling` pass before submission (Step 4b), and submitted with the runtime's own DD disabled.

**What changed vs. the parent notebook**
- `N` in `{3, 5}` (no `N=7`)
- `DD_MODES = ["off", "on"]` becomes `DD_METHODS = ["off", "XpXm", "XX", "XY4", "UDD"]`
- new **Step 4b** builds the UDD-padded circuits once per golden path
- plots and the consolidated save are keyed on `dd_method` instead of `dd_mode`
- a closing summary ranks the methods by mean deviation from theory

**Everything else is inherited unchanged** from the parent notebook: `K=2`, `T=1`, `N_RANDOM=5000`, `SHOTS=3`, the Pauli-twirling noise model, the pre-transpiled circuits (loaded from disk), per-lambda CSV/JSON checkpointing, and the `["QPA", ...]` job tags.

> **QPU cost.** Five methods instead of two is roughly 2.5x the parent notebook's job count (~300 jobs for `N` in `{3,5}` x 10 lambda x 2-4 batches). Each (N, method) pair is checkpointed independently, so an interrupted run resumes where it left off.

Pipeline:
1. Configuration
2. Backend setup (`ibm_pittsburgh`)
3. Circuit generation (unrolled)
4. Load pre-transpiled circuits
4b. **Build UDD-padded circuits (manual DD)**
5. Noise application (Pauli twirling)
6. Submission with per-lambda checkpointing -- one lambda sweep per DD method
7. Plot vs. theory -- all methods overlaid, plus residuals
8. Result inspection
9. Consolidated save + best-method ranking

In [2]:
import json
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from qiskit import qpy
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import XGate, RZGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
from qiskit_ibm_runtime import SamplerV2 as IBMSampler

from core.circuit_factory import CircuitFactory
from core.noise_models import PauliTwirlingStrategy
from execution.backend_handler import IBMRuntimeHandler
from analysis.result_processor import ResultProcessor

print("Imports loaded.")

Imports loaded.


## Step 1: Configuration

| Parameter | Value |
|-----------|-------|
| `N` | 3, 5 |
| `K` | 2 |
| `T` | 1 |
| `LAMBDA_PHASE` | `"even"` / `"odd"` / `"all"` |
| `N_RANDOM` | 5000 |
| `SHOTS` | 3 |
| `BATCH_SIZE` | 5000 |
| `DD_METHODS` | `["off", "XpXm", "XX", "XY4", "UDD"]` |
| `UDD_N_PULSES` | 4 |
| `TRANSPILE_SEEDS` | 10 |
| `DEVICE` | `ibm_pittsburgh` |

**DD method axis.** The parent notebook's `(dd_mode, dd_sequence)` pair is replaced by a single `DD_METHODS` list. Each entry is one full lambda sweep:
- `"off"` -- no DD (baseline).
- `"XpXm"`, `"XX"`, `"XY4"` -- applied server-side by the IBM runtime via `sampler.options.dynamical_decoupling`. These are the only three `sequence_type` values the runtime accepts.
- `"UDD"` -- Uhrig DD, applied client-side (Step 4b). The runtime's DD is disabled for this method so the circuit is not decoupled twice.

**Why UDD = 4 pulses.** UDD places X pulses at non-uniform times `t_j/T = sin^2(j*pi/(2n+2))`; an n-pulse train suppresses dephasing to order n. The count must be **even** so the X-train composes to the identity. A larger n needs a longer idle window to fit -- for these QPA circuits, n=4 lands a full train in essentially every idle window the scheduler exposes, whereas n=8 leaves many windows undecoupled. Step 4b prints the actual per-path pulse count so this can be retuned.

**Job tags.** Every `sampler.run(...)` is tagged `["QPA", "n={n}_k={K}_lambda={lambda}_dd={method}_batch={i}of{m}"]`. The `"QPA"` tag is the project-wide usage filter; the second tag now also carries the DD method, so jobs are separable per method straight from the IBM Quantum dashboard.

In [3]:
# ----- Experiment Parameters (inherited from the parent notebook) -----
K = 2                        # Qubits per register (d = 2^K = 4)
T = 1                        # Number of QPA trial rounds
N_RANDOM = 5000              # Pauli twirling instances per lambda
SHOTS = 3                    # Shots per circuit instance
BATCH_SIZE = 5000            # Circuits per IBM job submission
TRANSPILE_SEEDS = 10         # Multi-seed transpilation, pick lowest 2Q depth
OPT_LEVEL = 3                # Transpiler optimization level
DEVICE = "ibm_pittsburgh"
NO_RESET = False

# ----- Dynamical decoupling method sweep -----
# The full lambda sweep is run once per method below. "XpXm" is the parent
# notebook's default; "off" is the no-DD baseline. The goal is to see which
# sequence best closes the gap to the closed-form theory curve.
#
#   off  -> no DD
#   XpXm -> runtime built-in  (parent notebook default)  | server-side
#   XX   -> runtime built-in                             | server-side
#   XY4  -> runtime built-in                             | server-side
#   UDD  -> manual Uhrig DD (see Step 4b)                | client-side
DD_METHODS = ["off", "XpXm", "XX", "XY4", "UDD"]
RUNTIME_DD_SEQUENCES = {"XpXm", "XX", "XY4"}   # applied server-side by the sampler
MANUAL_DD_METHODS = {"UDD"}                     # applied client-side by a transpiler pass
DD_SKIP_RESET_QUBITS = True   # avoid pulsing qubits that are mid-reset

# UDD (Uhrig Dynamical Decoupling): X-only pulses at non-uniform times
#   t_j / T = sin^2( j*pi / (2n+2) ),  j = 1..n
# X is a native gate on Heron, so no basis translation is needed. n must be
# even so the pulse train composes to the identity. See Step 4b.
UDD_N_PULSES = 4
assert UDD_N_PULSES % 2 == 0, "UDD_N_PULSES must be even (the X-train must equal the identity)"

# ----- Lambda phase selector -----
LAMBDA_PHASE = "all"
LAMBDA_SETS = {
    "even": np.round(np.array([0.0, 0.2, 0.4, 0.6, 0.8]), 4),
    "odd":  np.round(np.array([0.1, 0.3, 0.5, 0.7, 0.9]), 4),
    "all":  np.round(np.arange(0.0, 1.0, 0.1), 4),
}
if LAMBDA_PHASE not in LAMBDA_SETS:
    raise ValueError(f"LAMBDA_PHASE must be one of {list(LAMBDA_SETS)}, got {LAMBDA_PHASE!r}")
lambdas = LAMBDA_SETS[LAMBDA_PHASE]

N_VALUES = [3, 5]
EXPERIMENTS = [{"n": n, "k": K, "t": T, "label": f"N={n}, K={K}, T={T}"} for n in N_VALUES]

# Output directory for incremental checkpoints (separate from the parent notebook's)
RESULTS_DIR = os.path.join("shared_data", "experiment_results", "end_to_end_hardware", "results", "end_to_end_unrolled_pittsburgh_t1_dd_methods")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"LAMBDA_PHASE = {LAMBDA_PHASE!r}")
print(f"Lambda sweep this pass ({len(lambdas)} points): {lambdas.tolist()}")
print(f"N_RANDOM={N_RANDOM}, SHOTS={SHOTS}, BATCH_SIZE={BATCH_SIZE}")
print(f"DD methods: {DD_METHODS}")
print(f"  runtime (server-side): {sorted(RUNTIME_DD_SEQUENCES)}")
print(f"  manual  (client-side): {sorted(MANUAL_DD_METHODS)}  (UDD_N_PULSES={UDD_N_PULSES})")
print(f"Experiments: {[e['label'] for e in EXPERIMENTS]}")
print(f"Results directory: {RESULTS_DIR}")

LAMBDA_PHASE = 'all'
Lambda sweep this pass (10 points): [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_RANDOM=5000, SHOTS=3, BATCH_SIZE=5000
DD methods: ['off', 'XpXm', 'XX', 'XY4', 'UDD']
  runtime (server-side): ['XX', 'XY4', 'XpXm']
  manual  (client-side): ['UDD']  (UDD_N_PULSES=4)
Experiments: ['N=3, K=2, T=1', 'N=5, K=2, T=1']
Results directory: shared_data/experiment_results/end_to_end_hardware/results/end_to_end_unrolled_pittsburgh_t1_dd_methods


## Step 2: Backend Setup

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()
backend = service.backend(DEVICE)

print(f"Backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Backend status: {backend.status()}")

Backend: ibm_pittsburgh
Number of qubits: 156
Backend status: <qiskit_ibm_runtime.models.backend_status.BackendStatus object at 0x116e77380>


## Step 3: Circuit Generation (Unrolled Strategy)

Number of unrolled paths per N (= 2^((N-1)/2) for T=1):
- N=3: 2 paths
- N=5: 4 paths

In [5]:
golden_data_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]

    strategy = CircuitFactory.create_strategy("unrolled", K, T, n, no_reset=NO_RESET)
    strategy.set_noise_strategy(None)
    golden_data = strategy.build(0.0)

    golden_circuits = [item['circuit'] for item in golden_data]
    golden_metadata = [{k: v for k, v in item.items() if k != 'circuit'} for item in golden_data]

    golden_data_per_exp[n] = {
        'circuits': golden_circuits,
        'metadata': golden_metadata,
        'strategy': strategy,
    }

    print(f"\n--- {label} ---")
    print(f"  Total unrolled paths: {len(golden_circuits)}")
    print(f"  Qubits/circuit: {golden_circuits[0].num_qubits}, Clbits/circuit: {golden_circuits[0].num_clbits}")


--- N=3, K=2, T=1 ---
  Total unrolled paths: 2
  Qubits/circuit: 7, Clbits/circuit: 3

--- N=5, K=2, T=1 ---
  Total unrolled paths: 4
  Qubits/circuit: 12, Clbits/circuit: 4


## Step 4: Load Pre-Transpiled Circuits

Best-of-N seed transpilation is done in [`find_best_transpilation_unrolled_pittsburgh_t1.ipynb`](find_best_transpilation_unrolled_pittsburgh_t1.ipynb) -- **the same transpilations the parent notebook uses.** Run that notebook first if the directory below is empty.

This cell loads the QPY files into `transpiled_per_exp[n]` and the per-path metadata into `transpile_summary_per_exp[n]`. The plain transpiled circuits are used directly for `off` and the three runtime DD methods; Step 4b derives the UDD-padded variants from them.

In [6]:
TRANSPILE_DIR = os.path.join("shared_data", "experiment_results", "end_to_end_hardware", "transpilations", "end_to_end_unrolled_pittsburgh_t1")

if not os.path.isdir(TRANSPILE_DIR):
    raise FileNotFoundError(
        f"Transpilation directory not found: {TRANSPILE_DIR}\n"
        f"Run `find_best_transpilation_unrolled_pittsburgh_t1.ipynb` first to populate it."
    )

transpiled_per_exp = {}
transpile_summary_per_exp = {}

for exp in EXPERIMENTS:
    n = exp["n"]
    label = exp["label"]
    nd = os.path.join(TRANSPILE_DIR, f"n{n}")
    summary_file = os.path.join(nd, "summary.json")

    if not os.path.exists(summary_file):
        raise FileNotFoundError(
            f"Missing summary for N={n}: {summary_file}\n"
            f"Run `find_best_transpilation_unrolled_pittsburgh_t1.ipynb` for this N."
        )

    with open(summary_file) as f:
        n_summary = json.load(f)

    # Order paths the same way the strategy emits them: path_0, path_1, ...
    golden_metadata = golden_data_per_exp[n]['metadata']
    path_order = [
        m.get('metadata', {}).get('path_name', f'path_{i}')
        for i, m in enumerate(golden_metadata)
    ]

    transpiled = []
    summary = []
    print(f"\n--- {label} ---")
    for path_name in path_order:
        if path_name not in n_summary.get('paths', {}):
            raise KeyError(
                f"Path {path_name!r} missing from {summary_file}. "
                f"Run the transpilation notebook to fill in this path."
            )
        path_meta = n_summary['paths'][path_name]
        qpy_file = os.path.join(nd, path_meta['qpy_file'])
        if not os.path.exists(qpy_file):
            raise FileNotFoundError(f"Missing QPY for {path_name}: {qpy_file}")

        with open(qpy_file, 'rb') as f:
            qcs = qpy.load(f)
        if len(qcs) != 1:
            raise ValueError(f"Expected 1 circuit in {qpy_file}, got {len(qcs)}")
        transpiled.append(qcs[0])

        summary.append({
            'path_name': path_name,
            'best_seed': path_meta['best_seed'],
            'best_2q_depth': path_meta['best_2q_depth'],
            'best_2q_gates': path_meta['best_2q_gates'],
            'depth': path_meta['depth'],
            'all_seed_2q_depths': [s['2q_depth'] for s in path_meta.get('all_seed_stats', [])],
        })
        print(f"  {path_name}: seed={path_meta['best_seed']}, 2Q depth={path_meta['best_2q_depth']}, "
              f"2Q gates={path_meta['best_2q_gates']}, gates={path_meta.get('gate_counts', {})}")

    transpiled_per_exp[n] = transpiled
    transpile_summary_per_exp[n] = summary

# Mirror the per-N summaries into the experiment results dir for traceability
combined_summary_path = os.path.join(RESULTS_DIR, "transpile_summary.json")
with open(combined_summary_path, 'w') as f:
    json.dump({str(n): transpile_summary_per_exp[n] for n in transpile_summary_per_exp}, f, indent=2)
print(f"\nTranspilation summary copy saved to: {combined_summary_path}")


--- N=3, K=2, T=1 ---
  path_0: seed=0, 2Q depth=18, 2Q gates=20, gates={'sx': 36, 'rz': 27, 'cz': 20, 'measure': 3, 'x': 1, 'reset': 1}
  path_1: seed=0, 2Q depth=18, 2Q gates=20, gates={'sx': 36, 'rz': 27, 'cz': 20, 'measure': 3, 'x': 1, 'reset': 1}

--- N=5, K=2, T=1 ---
  path_0: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'reset': 2, 'x': 1}
  path_1: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'reset': 2, 'x': 1}
  path_2: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'reset': 2, 'x': 1}
  path_3: seed=0, 2Q depth=18, 2Q gates=40, gates={'sx': 72, 'rz': 54, 'cz': 40, 'measure': 4, 'reset': 2, 'x': 1}

Transpilation summary copy saved to: shared_data/experiment_results/end_to_end_hardware/results/end_to_end_unrolled_pittsburgh_t1_dd_methods/transpile_summary.json


## Step 4b: Build UDD-Padded Circuits (Manual Dynamical Decoupling)

`XX`, `XpXm`, `XY4` are inserted by the IBM runtime at submission time. **UDD is not a runtime option**, so we pad it in ourselves.

For each golden transpiled circuit we run a two-pass `PassManager`:
1. `ALAPScheduleAnalysis` -- schedule the circuit against the backend `Target` (as-late-as-possible), exposing every idle window.
2. `PadDynamicalDecoupling` -- fill each idle window with the UDD X-pulse train.

The UDD sequence is `UDD_N_PULSES` X gates placed at the non-uniform Uhrig times via a custom `spacing` (the n+1 gaps around the pulses, as fractions of the idle window). X is a native Heron gate, so no basis translation is needed.

**This runs once per golden path** (2 paths for N=3, 4 for N=5). The padded circuit is then `.copy()`-ed per twirling instance downstream, exactly like the plain transpiled circuit -- so UDD adds no per-instance cost. `PassManager.run` drops the `TranspileLayout`, so it is restored onto each result; the downstream noise injection reads `qc.layout.initial_layout`.

`PadDynamicalDecoupling` degrades gracefully: an idle window too short to fit the whole train is left as a plain delay. The cell prints the X-pulse count actually inserted per path -- if it is low, lower `UDD_N_PULSES`. The runtime's own DD is disabled for the UDD method (Step 6) so the circuit is not decoupled twice.

In [7]:
def udd_spacing(n_pulses):
    """Relative gap fractions for an n-pulse Uhrig DD sequence.

    Pulse j sits at normalized time delta_j = sin^2(j*pi / (2n+2)). The
    transpiler pass wants the n+1 gaps around the pulses; they sum to 1.
    """
    deltas = [np.sin(np.pi * j / (2 * n_pulses + 2)) ** 2 for j in range(1, n_pulses + 1)]
    bounds = [0.0] + deltas + [1.0]
    return [bounds[i + 1] - bounds[i] for i in range(n_pulses + 1)]


def build_udd_circuits(transpiled_golden, backend, n_pulses, skip_reset_qubits=True):
    """Pad each transpiled golden circuit with an n-pulse Uhrig DD sequence.

    Returns new circuits carrying explicit Delay + X instructions in their idle
    windows. The TranspileLayout is restored onto each result so the downstream
    noise injection (which reads `qc.layout.initial_layout`) keeps working.
    """
    dd_sequence = [XGate()] * n_pulses
    spacing = udd_spacing(n_pulses)
    pm = PassManager([
        ALAPScheduleAnalysis(target=backend.target),
        PadDynamicalDecoupling(
            target=backend.target,
            dd_sequence=dd_sequence,
            spacing=spacing,
            skip_reset_qubits=skip_reset_qubits,
        ),
    ])
    udd_circuits = []
    for qc in transpiled_golden:
        udd_qc = pm.run(qc)
        udd_qc._layout = qc.layout   # PassManager drops .layout; restore for noise injection
        udd_circuits.append(udd_qc)
    return udd_circuits


udd_transpiled_per_exp = {}

if "UDD" in DD_METHODS:
    _spacing = udd_spacing(UDD_N_PULSES)
    print(f"UDD: {UDD_N_PULSES}-pulse Uhrig sequence (X-only)")
    print(f"  Uhrig spacing fractions (sum={sum(_spacing):.6f}): {[round(s, 4) for s in _spacing]}")
    for exp in EXPERIMENTS:
        n = exp["n"]
        udd_circuits = build_udd_circuits(
            transpiled_per_exp[n], backend, UDD_N_PULSES,
            skip_reset_qubits=DD_SKIP_RESET_QUBITS,
        )
        udd_transpiled_per_exp[n] = udd_circuits
        print(f"\n--- {exp['label']} ---")
        added_total = 0
        for i, (plain, udd) in enumerate(zip(transpiled_per_exp[n], udd_circuits)):
            added_x = udd.count_ops().get("x", 0) - plain.count_ops().get("x", 0)
            added_total += added_x
            n_windows = added_x // UDD_N_PULSES if UDD_N_PULSES else 0
            print(f"  path_{i}: +{added_x} X pulse(s) over {n_windows} idle window(s), "
                  f"depth {plain.depth()} -> {udd.depth()}")
        if added_total == 0:
            print(f"  WARNING: no UDD pulses fit any idle window for N={n}. "
                  f"UDD will behave like 'off' -- lower UDD_N_PULSES.")
else:
    print("UDD not in DD_METHODS -- skipping manual DD build.")

UDD: 4-pulse Uhrig sequence (X-only)
  Uhrig spacing fractions (sum=1.000000): [np.float64(0.0955), np.float64(0.25), np.float64(0.309), np.float64(0.25), np.float64(0.0955)]

--- N=3, K=2, T=1 ---
  path_0: +56 X pulse(s) over 14 idle window(s), depth 62 -> 82
  path_1: +48 X pulse(s) over 12 idle window(s), depth 62 -> 73

--- N=5, K=2, T=1 ---
  path_0: +104 X pulse(s) over 26 idle window(s), depth 61 -> 82
  path_1: +104 X pulse(s) over 26 idle window(s), depth 61 -> 82
  path_2: +104 X pulse(s) over 26 idle window(s), depth 61 -> 82
  path_3: +96 X pulse(s) over 24 idle window(s), depth 61 -> 73


## Step 5: Noise Application & Circuit Instance Generation

Identical fast-path twirling as the parent notebook: copy the transpiled circuit, sample Pauli noise, and inject ISA-safe gates at the front. For the UDD method the "transpiled" circuit passed in is the UDD-padded one from Step 4b; the noise injection is otherwise unchanged.

In [8]:
def build_noisy_batch(transpiled_golden, golden_circuits, golden_metadata, epsilon, n_random, batch_size):
    """Build batches of noisy circuit instances from pre-transpiled golden circuits."""
    all_batches = []
    batch_circuits = []
    batch_metadata = []

    for _ in range(n_random):
        noise_strategy = PauliTwirlingStrategy(K)

        for i, qc_transpiled in enumerate(transpiled_golden):
            qc_instance = qc_transpiled.copy()
            orig_qc = golden_circuits[i]

            data_regs = [reg for reg in orig_qc.qregs if reg.name.startswith("R")]
            noise_ops = noise_strategy.generate_noise_ops(data_regs, epsilon)

            layout = qc_transpiled.layout.initial_layout if qc_transpiled.layout else None

            for gate, logical_qubit in noise_ops:
                target_qubit = None
                if layout and logical_qubit in layout:
                    phys_qubit_idx = layout[logical_qubit]
                    target_qubit = qc_instance.qubits[phys_qubit_idx]
                elif logical_qubit in qc_instance.qubits:
                    target_qubit = logical_qubit
                else:
                    for q in qc_instance.qubits:
                        if hasattr(q, 'register') and hasattr(logical_qubit, 'register'):
                            if q.register.name == logical_qubit.register.name and q.index == logical_qubit.index:
                                target_qubit = q
                                break

                if target_qubit:
                    if gate.name == 'z':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                    elif gate.name == 'y':
                        qc_instance.data.insert(0, CircuitInstruction(RZGate(np.pi), (target_qubit,), ()))
                        qc_instance.data.insert(0, CircuitInstruction(XGate(), (target_qubit,), ()))
                    else:
                        qc_instance.data.insert(0, CircuitInstruction(gate, (target_qubit,), ()))

            batch_circuits.append(qc_instance)
            batch_metadata.append(golden_metadata[i])

            if len(batch_circuits) >= batch_size:
                all_batches.append((batch_circuits, batch_metadata))
                batch_circuits = []
                batch_metadata = []

    if batch_circuits:
        all_batches.append((batch_circuits, batch_metadata))

    return all_batches

print("build_noisy_batch() defined.")

build_noisy_batch() defined.


## Step 6: Submission with Incremental Checkpointing -- One Lambda Sweep per DD Method

For each `N`, the full lambda sweep is run once per method in `DD_METHODS`. Each (N, method) pair has its own checkpoint pair:

```
results_n{N}_k{K}_t{T}_unrolled_ibm_pittsburgh_dd_{method}.{csv,json}
```

After **every lambda point** the partial result list is written to disk; re-running the cell skips lambdas already in the CSV.

**How each method is applied:**
- `off` -- `sampler.options.dynamical_decoupling.enable = False`, plain transpiled circuits.
- `XpXm` / `XX` / `XY4` -- `enable = True`, `sequence_type = <method>`, plain transpiled circuits (the runtime inserts the DD).
- `UDD` -- `enable = False` (the DD is already padded in), and the **UDD-padded circuits from Step 4b** are submitted instead of the plain ones.

Each `sampler.run(...)` gets a freshly built sampler so the per-job tag and DD options reflect the current (N, lambda, method).

In [9]:
def checkpoint_paths(n, dd_method):
    base = f"results_n{n}_k{K}_t{T}_unrolled_ibm_pittsburgh_dd_{dd_method}"
    return (
        os.path.join(RESULTS_DIR, base + ".csv"),
        os.path.join(RESULTS_DIR, base + ".json"),
    )

def load_checkpoint(n, dd_method):
    csv_path, _ = checkpoint_paths(n, dd_method)
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        return df.to_dict('records')
    return []

def save_checkpoint(n, dd_method, results, meta):
    csv_path, json_path = checkpoint_paths(n, dd_method)
    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w') as f:
        json.dump({'meta': meta, 'results': results}, f, indent=2)

def build_tagged_sampler(backend, n, epsilon, dd_method, batch_idx, n_batches):
    """Construct a SamplerV2 with job tags and DD options for the current (N, lambda, method)."""
    sampler = IBMSampler(mode=backend)

    # Job tags -- first tag is the project-wide filter, second encodes job params.
    param_tag = (f"n={n}_k={K}_lambda={float(epsilon):.4f}"
                 f"_dd={dd_method}_batch={batch_idx+1}of{n_batches}")
    sampler.options.environment.job_tags = ["QPA", param_tag]

    # Dynamical decoupling:
    #   runtime sequences -> let the runtime insert the DD
    #   "off"             -> no DD
    #   "UDD" (manual)    -> DD already padded into the circuit, so the runtime
    #                        DD must stay OFF or the circuit is decoupled twice
    if dd_method in RUNTIME_DD_SEQUENCES:
        sampler.options.dynamical_decoupling.enable = True
        sampler.options.dynamical_decoupling.sequence_type = dd_method
        sampler.options.dynamical_decoupling.skip_reset_qubits = DD_SKIP_RESET_QUBITS
    else:
        sampler.options.dynamical_decoupling.enable = False

    return sampler

def run_experiment(n, dd_method, transpiled_golden, golden_circuits, golden_metadata,
                   lambdas, n_random, shots_per_circuit, batch_size, backend):
    """Lambda sweep for one (N, dd_method). Resumes from CSV checkpoint if it exists.

    `transpiled_golden` is the plain transpiled set for the runtime/off methods,
    or the UDD-padded set for the manual UDD method (selected by the caller).
    """
    result_processor = ResultProcessor(K)

    results = load_checkpoint(n, dd_method)
    done_lambdas = {round(float(r['lambda']), 4) for r in results}
    if done_lambdas:
        print(f"  Resuming N={n}, dd={dd_method}: {len(done_lambdas)} lambda point(s) "
              f"already in checkpoint -> {sorted(done_lambdas)}")

    is_manual = dd_method in MANUAL_DD_METHODS
    meta = {
        'n': n, 'k': K, 't': T,
        'dd_method': dd_method,
        'dd_kind': ('manual' if is_manual
                    else 'runtime' if dd_method in RUNTIME_DD_SEQUENCES else 'none'),
        'udd_n_pulses': UDD_N_PULSES if dd_method == 'UDD' else None,
        'dd_skip_reset_qubits': DD_SKIP_RESET_QUBITS if dd_method != 'off' else None,
        'n_random': n_random, 'shots_per_circuit': shots_per_circuit,
        'batch_size': batch_size, 'backend': backend.name,
        'transpile_seeds': TRANSPILE_SEEDS, 'opt_level': OPT_LEVEL,
        'job_tags_project': 'QPA',
    }

    for epsilon in tqdm(lambdas, desc=f"N={n} dd={dd_method} lambda sweep"):
        eps_key = round(float(epsilon), 4)
        if eps_key in done_lambdas:
            continue

        batches = build_noisy_batch(
            transpiled_golden, golden_circuits, golden_metadata,
            epsilon, n_random, batch_size,
        )
        n_batches = len(batches)

        global_path_stats = defaultdict(lambda: {'success': 0, 'total': 0})
        submitted_job_ids = []

        for batch_idx, (batch_circs, batch_meta) in enumerate(batches):
            sampler = build_tagged_sampler(backend, n, epsilon, dd_method, batch_idx, n_batches)
            pubs = [(qc, None, shots_per_circuit) for qc in batch_circs]
            try:
                job = sampler.run(pubs)
                submitted_job_ids.append(job.job_id())
                pub_result = job.result()

                extracted_counts = ResultProcessor.extract_counts_from_job_result(pub_result, is_dynamic=False)

                total_clbits_list = []
                for counts in extracted_counts:
                    if counts:
                        first_key = next(iter(counts))
                        total_clbits_list.append(len(first_key.replace(" ", "")))
                    else:
                        total_clbits_list.append(0)

                batch_stats = result_processor.aggregate_batch_stats(
                    extracted_counts, batch_meta, total_clbits_list
                )
                for cond_key, stats in batch_stats.items():
                    global_path_stats[cond_key]['success'] += stats['success']
                    global_path_stats[cond_key]['total'] += stats['total']

            except Exception as e:
                print(f"  Error at lambda={epsilon:.4f}, batch {batch_idx}: {e}")
                import traceback
                traceback.print_exc()

        fidelity = 0.0
        for stats in global_path_stats.values():
            if stats['total'] > 0:
                fidelity += stats['success'] / stats['total']

        results.append({
            'lambda': float(epsilon),
            'fidelity': float(fidelity),
            'job_ids': ",".join(submitted_job_ids),
            'n_batches': n_batches,
        })
        results.sort(key=lambda r: r['lambda'])
        save_checkpoint(n, dd_method, results, meta)
        done_lambdas.add(eps_key)
        print(f"  N={n} dd={dd_method}, lambda={epsilon:.4f} -> fidelity={fidelity:.4f}  "
              f"[{n_batches} batch(es), checkpoint saved]")

    return results

print("run_experiment() defined.")

run_experiment() defined.


### Run All Experiments (N in {3, 5} x DD method)

Loops over every (N, method) pair. For `UDD` the loop feeds in the UDD-padded circuits from Step 4b; every other method uses the plain transpiled circuits. Each pair is checkpointed independently -- if the kernel dies, re-run the cell to resume.

In [ ]:
results_per_exp = {n: {} for n in N_VALUES}
df_per_exp = {n: {} for n in N_VALUES}

for exp in EXPERIMENTS:
    n = exp["n"]
    for dd_method in DD_METHODS:
        print(f"\n========== {exp['label']}  |  dd={dd_method} ==========")

        # UDD submits the pre-padded circuits; every other method uses the plain ones.
        if dd_method in MANUAL_DD_METHODS:
            transpiled_golden = udd_transpiled_per_exp[n]
        else:
            transpiled_golden = transpiled_per_exp[n]

        results_nd = run_experiment(
            n=n,
            dd_method=dd_method,
            transpiled_golden=transpiled_golden,
            golden_circuits=golden_data_per_exp[n]['circuits'],
            golden_metadata=golden_data_per_exp[n]['metadata'],
            lambdas=lambdas,
            n_random=N_RANDOM,
            shots_per_circuit=SHOTS,
            batch_size=BATCH_SIZE,
            backend=backend,
        )
        results_per_exp[n][dd_method] = results_nd
        df_per_exp[n][dd_method] = pd.DataFrame(results_nd)
        print(f"N={n} dd={dd_method}: {len(df_per_exp[n][dd_method])} lambda points collected")

for n in sorted(df_per_exp):
    for dd_method in DD_METHODS:
        df = df_per_exp[n].get(dd_method)
        if df is None or len(df) == 0:
            continue
        print(f"\n--- N={n}, dd={dd_method} ---")
        cols = [c for c in ['lambda', 'fidelity', 'n_batches'] if c in df.columns]
        print(df[cols].to_string(index=False))


========== N=3, K=2, T=1  |  dd=off ==========
  Resuming N=3, dd=off: 10 lambda point(s) already in checkpoint -> [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


N=3 dd=off lambda sweep:   0%|          | 0/10 [00:00<?, ?it/s]

N=3 dd=off: 10 lambda points collected

========== N=3, K=2, T=1  |  dd=XpXm ==========
  Resuming N=3, dd=XpXm: 10 lambda point(s) already in checkpoint -> [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


N=3 dd=XpXm lambda sweep:   0%|          | 0/10 [00:00<?, ?it/s]

N=3 dd=XpXm: 10 lambda points collected

========== N=3, K=2, T=1  |  dd=XX ==========
  Resuming N=3, dd=XX: 4 lambda point(s) already in checkpoint -> [0.0, 0.1, 0.2, 0.3]


N=3 dd=XX lambda sweep:   0%|          | 0/10 [00:00<?, ?it/s]

/Users/henryzou/.venvs/opqa/lib/python3.13/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1001: UserWarning: The backend ibm_pittsburgh currently has a status of maintenance.
  warnings.warn(


## Step 7: Plot -- Experimental vs. Theory, All DD Methods

Theory curves for K=2 (d=4):
- N=3: $F(\lambda) = \tfrac{1}{8}(8 - 2\lambda - 7\lambda^2 + 3\lambda^3)$
- N=5: $F(\lambda) = \tfrac{1}{640}(640 - 96\lambda - 224\lambda^2 - 700\lambda^3 + 693\lambda^4 - 153\lambda^5)$

One column per `N`. The **top row** overlays every DD method's fidelity-decay curve on the closed-form theory; the **bottom row** shows the residual (experiment - theory) for each method -- the method whose residual sits closest to zero is the one that best preserves fidelity.

In [ ]:
def theory_curve(lam, n, k):
    """Closed-form theory for K=2, available for N=3,5,7."""
    if k == 2:
        if n == 3:
            return (1/8) * (8 - 2*lam - 7*lam**2 + 3*lam**3)
        elif n == 5:
            return (1/640) * (640 - 96*lam - 224*lam**2 - 700*lam**3 + 693*lam**4 - 153*lam**5)
        elif n == 7:
            return (1/215040) * (215040 - 23040*lam - 50688*lam**2 - 88768*lam**3
                                  - 311056*lam**4 + 497192*lam**5 - 207951*lam**6 + 23031*lam**7)
    return None

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 15,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10,
    'lines.linewidth': 2, 'lines.markersize': 7,
})

# One fixed color + marker per DD method so the two N columns stay comparable.
method_style = {
    'off':  dict(color='#9467bd', marker='o'),
    'XpXm': dict(color='#1f77b4', marker='s'),
    'XX':   dict(color='#2ca02c', marker='^'),
    'XY4':  dict(color='#d62728', marker='D'),
    'UDD':  dict(color='#ff7f0e', marker='P'),
}

n_values_sorted = sorted(df_per_exp.keys())
lam_fine = np.linspace(0, 1, 200)

fig, axes = plt.subplots(2, len(n_values_sorted),
                         figsize=(8 * len(n_values_sorted), 11), squeeze=False)

for col, n in enumerate(n_values_sorted):
    ax_fid = axes[0][col]
    ax_res = axes[1][col]
    th_fine = theory_curve(lam_fine, n, K)

    # --- Top: fidelity decay ---
    if th_fine is not None:
        ax_fid.plot(lam_fine, th_fine, '--', color='black', linewidth=1.4, alpha=0.7,
                    label='Theory (closed form)')
    for dd_method in DD_METHODS:
        df = df_per_exp[n].get(dd_method)
        if df is None or len(df) == 0:
            continue
        style = method_style.get(dd_method, dict(color=None, marker='o'))
        ax_fid.plot(df['lambda'], df['fidelity'], style['marker'] + '-',
                    color=style['color'], alpha=0.9, label=f'dd = {dd_method}')
    ax_fid.axhline(y=0.25, color='gray', linestyle=':', alpha=0.4,
                   label='Random guess (1/d = 0.25)')
    ax_fid.set_title(f'N={n}, K={K}, T={T} -- Fidelity decay', fontweight='bold')
    ax_fid.set_xlabel(r'Depolarizing Noise Strength ($\lambda$)')
    ax_fid.set_ylabel('Purified Fidelity')
    ax_fid.set_xlim(-0.02, 1.02)
    ax_fid.set_ylim(0.20, 1.05)
    ax_fid.legend(loc='upper right', frameon=True)
    ax_fid.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    # --- Bottom: residual vs theory ---
    ax_res.axhline(y=0.0, color='black', linestyle='--', linewidth=1.4, alpha=0.7)
    for dd_method in DD_METHODS:
        df = df_per_exp[n].get(dd_method)
        if df is None or len(df) == 0:
            continue
        th = theory_curve(df['lambda'].to_numpy(), n, K)
        if th is None:
            continue
        style = method_style.get(dd_method, dict(color=None, marker='o'))
        ax_res.plot(df['lambda'], df['fidelity'].to_numpy() - th, style['marker'] + '-',
                    color=style['color'], alpha=0.9, label=f'dd = {dd_method}')
    ax_res.set_title(f'N={n} -- Residual (experiment - theory)', fontweight='bold')
    ax_res.set_xlabel(r'Depolarizing Noise Strength ($\lambda$)')
    ax_res.set_ylabel('Fidelity - Theory')
    ax_res.set_xlim(-0.02, 1.02)
    ax_res.legend(loc='lower right', frameon=True)
    ax_res.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

fig.suptitle('QPA Fidelity Decay -- DD Method Comparison -- Unrolled (T=1) -- IBM Pittsburgh',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()

plot_path = os.path.join(RESULTS_DIR, "fidelity_decay_dd_methods_ibm_pittsburgh.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")
plt.show()

## Step 8: Result Inspection

In [ ]:
for n in sorted(df_per_exp):
    for dd_method in DD_METHODS:
        df = df_per_exp[n].get(dd_method)
        if df is None or len(df) == 0:
            continue
        print(f"\n=== N={n}, K={K}, T={T}, dd={dd_method} ===")
        print(f"{'Lambda':>8} {'Exp Fidelity':>14} {'Theory':>12} {'Delta':>10}")
        print("-" * 48)
        for _, row in df.iterrows():
            lam = row['lambda']
            exp_f = row['fidelity']
            th_f = theory_curve(lam, n, K)
            if th_f is not None:
                delta = exp_f - th_f
                print(f"{lam:8.4f} {exp_f:14.4f} {th_f:12.4f} {delta:+10.4f}")
            else:
                print(f"{lam:8.4f} {exp_f:14.4f} {'n/a':>12} {'n/a':>10}")

## Step 9: Consolidated Save + Best-Method Ranking

Per-(N, method) CSV/JSON files are written incrementally during the sweep. This step combines them into one consolidated CSV + JSON, then ranks the DD methods per `N` by **mean absolute deviation from theory** (lower = closer to the ideal curve = better).

In [ ]:
combined_rows = []
for n in sorted(df_per_exp):
    for dd_method in DD_METHODS:
        df = df_per_exp[n].get(dd_method)
        if df is None or len(df) == 0:
            continue
        for _, row in df.iterrows():
            lam = float(row['lambda'])
            exp_f = float(row['fidelity'])
            th_f = theory_curve(lam, n, K)
            combined_rows.append({
                'n': n, 'k': K, 't': T,
                'dd_method': dd_method,
                'lambda': lam,
                'fidelity_experiment': exp_f,
                'fidelity_theory': float(th_f) if th_f is not None else None,
                'delta': (exp_f - float(th_f)) if th_f is not None else None,
                'job_ids': row.get('job_ids', ''),
                'n_batches': row.get('n_batches', None),
            })

combined_df = pd.DataFrame(combined_rows)

combined_csv = os.path.join(RESULTS_DIR, "results_all_dd_methods_ibm_pittsburgh.csv")
combined_json = os.path.join(RESULTS_DIR, "results_all_dd_methods_ibm_pittsburgh.json")
combined_df.to_csv(combined_csv, index=False)

# --- Rank DD methods per N by mean |delta| from theory ---
ranking_rows = []
for n in sorted(df_per_exp):
    sub = combined_df[(combined_df['n'] == n) & combined_df['delta'].notna()]
    for dd_method in DD_METHODS:
        msub = sub[sub['dd_method'] == dd_method]
        if len(msub) == 0:
            continue
        ranking_rows.append({
            'n': n,
            'dd_method': dd_method,
            'mean_abs_delta': float(msub['delta'].abs().mean()),
            'mean_delta': float(msub['delta'].mean()),
            'n_lambda_points': int(len(msub)),
        })
ranking_df = pd.DataFrame(ranking_rows).sort_values(['n', 'mean_abs_delta'])

run_meta = {
    'backend': backend.name,
    'k': K, 't': T,
    'n_values': sorted(df_per_exp.keys()),
    'dd_methods': DD_METHODS,
    'runtime_dd_sequences': sorted(RUNTIME_DD_SEQUENCES),
    'manual_dd_methods': sorted(MANUAL_DD_METHODS),
    'udd_n_pulses': UDD_N_PULSES,
    'dd_skip_reset_qubits': DD_SKIP_RESET_QUBITS,
    'lambdas': lambdas.tolist(),
    'n_random': N_RANDOM,
    'shots_per_circuit': SHOTS,
    'batch_size': BATCH_SIZE,
    'transpile_seeds': TRANSPILE_SEEDS,
    'opt_level': OPT_LEVEL,
    'job_tags_project': 'QPA',
}
with open(combined_json, 'w') as f:
    json.dump({'meta': run_meta, 'results': combined_rows,
               'ranking': ranking_rows}, f, indent=2)

print(f"Combined CSV  -> {combined_csv}")
print(f"Combined JSON -> {combined_json}")

print("\n=== DD method ranking (mean |experiment - theory|, lower is better) ===")
for n in sorted(df_per_exp):
    nsub = ranking_df[ranking_df['n'] == n]
    if len(nsub) == 0:
        continue
    print(f"\n--- N={n} ---")
    print(f"{'rank':>4} {'dd_method':>10} {'mean|delta|':>12} {'mean delta':>12}")
    for rank, (_, r) in enumerate(nsub.iterrows(), start=1):
        best = "  <- best" if rank == 1 else ""
        print(f"{rank:>4} {r['dd_method']:>10} {r['mean_abs_delta']:>12.4f} "
              f"{r['mean_delta']:>+12.4f}{best}")

combined_df

noisy t-gate prep, purifiy the state

Question: even if you purify it, if you don't embed it with error correcting code, how can we protect it and know that our output carried is good?